# CatBoost Default — CPU vs GPU Precision Test

CatBoost GPU uses **float32** for histogram building; CPU uses **float64**.  
Reduced precision can mean slightly different split points across 1000+ trees.  
This tests whether CPU produces a meaningfully different (higher?) LB score.

**Reference**: GPU seed=42 → OOF AUC 0.95540, LB 0.95356  
All other params identical: iterations=2000, lr=0.05, depth=6, early_stopping=50

In [1]:
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
FEATURES = [c for c in train.columns if c not in ['heart_disease', 'id']]

X      = train[FEATURES]
y      = train['heart_disease'].values
X_test = test[FEATURES]

print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


In [2]:
params = dict(
    iterations        = 2000,
    learning_rate     = 0.05,
    depth             = 6,
    task_type         = 'CPU',
    thread_count      = -1,
    cat_features      = CAT_FEATURES,
    random_seed       = 42,
    verbose           = 100,  # print every 100 iters to monitor progress
)

skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof        = np.zeros(len(y))
test_folds = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')
    m = CatBoostClassifier(**params)
    m.fit(X.iloc[tr_idx], y[tr_idx],
          eval_set=(X.iloc[val_idx], y[val_idx]),
          early_stopping_rounds=50)
    oof[val_idx]     = m.predict_proba(X.iloc[val_idx])[:, 1]
    test_folds[fold] = m.predict_proba(X_test)[:, 1]
    print(f'Fold {fold + 1}  AUC={roc_auc_score(y[val_idx], oof[val_idx]):.5f}'
          f'  best_iter={m.best_iteration_}')

cv_auc     = roc_auc_score(y, oof)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (CPU): {cv_auc:.5f}')
print(f'OOF AUC (GPU): 0.95540  (reference)')
print(f'Delta:         {cv_auc - 0.95540:+.5f}')


=== Fold 1/5 ===
0:	learn: 0.6421484	test: 0.6420187	best: 0.6420187 (0)	total: 179ms	remaining: 5m 58s
100:	learn: 0.2725554	test: 0.2716219	best: 0.2716219 (100)	total: 4.19s	remaining: 1m 18s
200:	learn: 0.2700229	test: 0.2696783	best: 0.2696783 (200)	total: 8.46s	remaining: 1m 15s
300:	learn: 0.2688081	test: 0.2688699	best: 0.2688699 (300)	total: 13s	remaining: 1m 13s
400:	learn: 0.2676763	test: 0.2679509	best: 0.2679509 (400)	total: 17.5s	remaining: 1m 9s
500:	learn: 0.2668648	test: 0.2674355	best: 0.2674355 (499)	total: 22.2s	remaining: 1m 6s
600:	learn: 0.2662688	test: 0.2671090	best: 0.2671090 (600)	total: 26.8s	remaining: 1m 2s
700:	learn: 0.2657843	test: 0.2669763	best: 0.2669735 (699)	total: 31.4s	remaining: 58.2s
800:	learn: 0.2653266	test: 0.2668315	best: 0.2668315 (800)	total: 36.1s	remaining: 54s
900:	learn: 0.2649119	test: 0.2667590	best: 0.2667590 (900)	total: 40.9s	remaining: 49.9s
1000:	learn: 0.2645375	test: 0.2667304	best: 0.2667303 (981)	total: 45.8s	remaining: 4

In [ ]:
import numpy as np

# Save OOF for stacking
np.save('submissions/oof_cat_cpu.npy', oof)
print(f'OOF saved: submissions/oof_cat_cpu.npy  (AUC={roc_auc_score(y, oof):.5f})')

label = 'catboost_default_cpu'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc={cv_auc:.4f}'
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')
print(f'desc: {desc}')